# 05 - Representation Weighting & Coverage Diagnostics

Tahap ini **tidak mengklaim sampel X mewakili penduduk Jakarta atau Banten**. Bobot hanya mengurangi ketimpangan yang dapat diamati di dalam desain pengambilan data: dominasi akun aktif, perbedaan jumlah hasil antarkueri, dan bobot sumber dari tahap verifikasi.

Kalibrasi populasi bersifat opsional dan hanya dijalankan jika `regional_reference.csv` berisi angka resmi, URL sumber, tahun referensi, dan `verified=True`. Kolom `_region` menunjukkan **strata kueri**, bukan domisili penulis.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
BOOTSTRAP_ITERATIONS = 500
LOWER_TRIM_QUANTILE = 0.01
UPPER_TRIM_QUANTILE = 0.99

SENTIMENT_PATHS = [
    Path('organized_csv/05_sentiment/sentiment_results.csv'),
    Path('sentiment_results.csv'),
]
LDA_PATHS = [
    Path('organized_csv/06_lda/lda_results.csv'),
    Path('lda_results.csv'),
]
REFERENCE_PATH = Path('regional_reference.csv')
OUTPUT_DIR = Path('organized_csv/07_weighting')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def first_existing(paths, required=True):
    match = next((p for p in paths if p.exists()), None)
    if match is None and required:
        raise FileNotFoundError(f'File input tidak ditemukan: {[str(p) for p in paths]}')
    return match

sentiment_path = first_existing(SENTIMENT_PATHS)
lda_path = first_existing(LDA_PATHS, required=False)
print(f'Input sentimen: {sentiment_path}')
print(f'Input topik   : {lda_path or "tidak tersedia (opsional)"}')

## 1. Validasi input

Baris tanpa identitas akun diberi ID sintetis per tweet agar tidak digabung sebagai satu akun. Analisis utama hanya memakai prediksi sentimen yang lolos review otomatis (`accepted`).

In [ ]:
df = pd.read_csv(sentiment_path, low_memory=False)
required_columns = {
    'id', 'author_username', '_region', '_search_keyword', 'source_weight',
    'sentiment_label', 'sentiment_value', 'sentiment_review_status'
}
missing = sorted(required_columns - set(df.columns))
if missing:
    raise ValueError(f'Kolom tahap 03 belum lengkap: {missing}')

df = df.copy()
df['id'] = df['id'].astype(str)
df['_region'] = df['_region'].astype(str).str.strip()
df['_search_keyword'] = df['_search_keyword'].astype(str).str.strip()
df['source_weight'] = pd.to_numeric(df['source_weight'], errors='coerce')
df['sentiment_value'] = pd.to_numeric(df['sentiment_value'], errors='coerce')

valid = df[
    df['sentiment_review_status'].eq('accepted')
    & df['sentiment_value'].notna()
    & df['source_weight'].gt(0)
    & df['_region'].ne('')
    & df['_search_keyword'].ne('')
].copy()

username = valid['author_username'].fillna('').astype(str).str.strip().str.lower()
valid['analysis_account_id'] = np.where(username.ne(''), username, 'missing_user__' + valid['id'])

print(f'Semua hasil sentimen : {len(df):,}')
print(f'Lolos analisis utama : {len(valid):,}')
print(f'Akun unik            : {valid.analysis_account_id.nunique():,}')
print(f'Strata wilayah kueri : {valid._region.nunique()}')
display(valid[['_region', '_search_keyword', 'sentiment_label', 'source_weight']].head())

## 2. Bangun bobot operasional

- `source_weight`: kualitas sumber dari tahap 02.
- `account_weight`: moderasi akun yang mengirim banyak tweet, menggunakan $1/\sqrt{n}$.
- `query_weight`: menyamakan kontribusi kueri di dalam setiap strata wilayah.
- Bobot gabungan dipangkas pada persentil 1 dan 99, lalu dinormalisasi agar rata-ratanya 1.

Bobot alternatif `one_account_weight` disediakan untuk uji sensitivitas yang memberi total kontribusi sama pada setiap akun.

In [ ]:
account_count = valid.groupby('analysis_account_id')['id'].transform('size')
valid['tweets_by_account'] = account_count
valid['account_weight'] = 1.0 / np.sqrt(account_count)
valid['one_account_weight'] = 1.0 / account_count

query_counts = valid.groupby(['_region', '_search_keyword'])['id'].transform('size')
queries_per_region = valid.groupby('_region')['_search_keyword'].transform('nunique')
rows_per_region = valid.groupby('_region')['id'].transform('size')
observed_query_share = query_counts / rows_per_region
target_query_share = 1.0 / queries_per_region
valid['query_weight'] = target_query_share / observed_query_share

valid['weight_source_only'] = valid['source_weight']
valid['weight_operational_raw'] = (
    valid['source_weight'] * valid['account_weight'] * valid['query_weight']
)
valid['weight_one_account_raw'] = (
    valid['source_weight'] * valid['one_account_weight'] * valid['query_weight']
)

def trim_and_normalize(series, lower=LOWER_TRIM_QUANTILE, upper=UPPER_TRIM_QUANTILE):
    series = pd.to_numeric(series, errors='coerce')
    lo, hi = series.quantile([lower, upper])
    trimmed = series.clip(lower=lo, upper=hi)
    return trimmed / trimmed.mean(), float(lo), float(hi)

valid['weight_operational'], op_lo, op_hi = trim_and_normalize(valid['weight_operational_raw'])
valid['weight_one_account'], oa_lo, oa_hi = trim_and_normalize(valid['weight_one_account_raw'])

print(f'Trimming operasional: [{op_lo:.4f}, {op_hi:.4f}]')
print(f'Trimming satu-akun  : [{oa_lo:.4f}, {oa_hi:.4f}]')
display(valid[['source_weight', 'account_weight', 'query_weight', 'weight_operational']].describe().T.round(3))

## 3. Diagnostik bobot dan cakupan

Effective sample size (ESS) mengukur berapa banyak observasi independen ekuivalen yang tersisa setelah pembobotan. Rasio ESS rendah menandakan bobot terlalu terkonsentrasi dan hasil perlu dibaca lebih hati-hati.

In [ ]:
def weight_diagnostic(name, weights):
    w = np.asarray(weights, dtype=float)
    ess = (w.sum() ** 2) / np.square(w).sum()
    cv = w.std(ddof=0) / w.mean()
    return {
        'weight_scheme': name,
        'n_tweets': len(w),
        'weight_min': w.min(),
        'weight_median': np.median(w),
        'weight_mean': w.mean(),
        'weight_max': w.max(),
        'coefficient_of_variation': cv,
        'design_effect_approx': 1 + cv ** 2,
        'effective_sample_size': ess,
        'ess_ratio': ess / len(w),
    }

weight_diagnostics = pd.DataFrame([
    weight_diagnostic('unweighted', np.ones(len(valid))),
    weight_diagnostic('source_only', valid['weight_source_only']),
    weight_diagnostic('operational', valid['weight_operational']),
    weight_diagnostic('one_account_sensitivity', valid['weight_one_account']),
])
display(weight_diagnostics.round(4))

def coverage_table(group_columns):
    grouped = valid.groupby(group_columns, dropna=False)
    out = grouped.agg(
        n_tweets=('id', 'size'),
        n_unique_accounts=('analysis_account_id', 'nunique'),
        median_tweets_per_account=('tweets_by_account', 'median'),
        mean_source_weight=('source_weight', 'mean'),
        sum_operational_weight=('weight_operational', 'sum'),
    ).reset_index()
    out['tweets_per_unique_account'] = out['n_tweets'] / out['n_unique_accounts']
    return out

coverage_by_region = coverage_table(['_region'])
coverage_by_keyword = coverage_table(['_region', '_search_keyword'])

account_contribution = valid.groupby('analysis_account_id').agg(
    n_tweets=('id', 'size'),
    unweighted_share=('id', lambda x: len(x) / len(valid)),
    operational_weight=('weight_operational', 'sum'),
).reset_index()
account_contribution['operational_share'] = (
    account_contribution['operational_weight'] / account_contribution['operational_weight'].sum()
)
display(coverage_by_region.round(3))
display(account_contribution.nlargest(10, 'n_tweets').round(4))

## 4. Estimasi sentimen dan sensitivitas

Hasil utama dibandingkan dalam empat skema. Perubahan kesimpulan yang besar antarskema adalah temuan sensitivitas, bukan alasan memilih skema yang paling sesuai harapan.

In [ ]:
WEIGHT_SCHEMES = {
    'unweighted': None,
    'source_only': 'weight_source_only',
    'operational': 'weight_operational',
    'one_account_sensitivity': 'weight_one_account',
}

def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return float(np.average(values, weights=weights))

def estimate_sentiment(data, group_name, group_value):
    rows = []
    for scheme, weight_col in WEIGHT_SCHEMES.items():
        weights = np.ones(len(data)) if weight_col is None else data[weight_col].to_numpy()
        total = weights.sum()
        rows.append({
            'group_type': group_name,
            'group_value': group_value,
            'weight_scheme': scheme,
            'n_tweets': len(data),
            'n_unique_accounts': data['analysis_account_id'].nunique(),
            'mean_sentiment': weighted_mean(data['sentiment_value'], weights),
            'negative_share': weights[data['sentiment_label'].eq('Negatif')].sum() / total,
            'neutral_share': weights[data['sentiment_label'].eq('Netral')].sum() / total,
            'positive_share': weights[data['sentiment_label'].eq('Positif')].sum() / total,
        })
    return rows

estimate_rows = estimate_sentiment(valid, 'overall', 'all')
for region, subset in valid.groupby('_region'):
    estimate_rows.extend(estimate_sentiment(subset, 'query_region', region))
for (region, keyword), subset in valid.groupby(['_region', '_search_keyword']):
    estimate_rows.extend(estimate_sentiment(subset, 'query_keyword', f'{region} | {keyword}'))

sentiment_estimates = pd.DataFrame(estimate_rows)
display(sentiment_estimates.query("group_type in ['overall', 'query_region']").round(4))

## 5. Cluster bootstrap per akun

Interval ketidakpastian dibuat dengan meresampling akun, bukan tweet. Ini menghormati ketergantungan antar-tweet dari akun yang sama. Interval hanya menggambarkan ketidakpastian pada sampel yang diamati, bukan bias cakupan platform.

In [ ]:
def cluster_bootstrap(data, weight_col='weight_operational', iterations=BOOTSTRAP_ITERATIONS, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    working = data.assign(
        _weighted_sentiment=data[weight_col] * data['sentiment_value'],
        _weighted_negative=data[weight_col] * data['sentiment_label'].eq('Negatif').astype(float),
    )
    account_stats = working.groupby('analysis_account_id', sort=False).agg(
        sum_weight=(weight_col, 'sum'),
        sum_weighted_sentiment=('_weighted_sentiment', 'sum'),
        sum_weighted_negative=('_weighted_negative', 'sum'),
    )
    values = account_stats.to_numpy(dtype=float)
    n_accounts = len(values)
    sampled_indices = rng.integers(0, n_accounts, size=(iterations, n_accounts))
    sampled_totals = values[sampled_indices].sum(axis=1)
    bootstrap_sentiment = sampled_totals[:, 1] / sampled_totals[:, 0]
    bootstrap_negative = sampled_totals[:, 2] / sampled_totals[:, 0]

    w = data[weight_col].to_numpy(dtype=float)
    return {
        'n_tweets': len(data),
        'n_unique_accounts': n_accounts,
        'bootstrap_iterations': iterations,
        'mean_sentiment': weighted_mean(data['sentiment_value'], w),
        'mean_sentiment_ci_low': np.quantile(bootstrap_sentiment, 0.025),
        'mean_sentiment_ci_high': np.quantile(bootstrap_sentiment, 0.975),
        'negative_share': np.average(data['sentiment_label'].eq('Negatif'), weights=w),
        'negative_share_ci_low': np.quantile(bootstrap_negative, 0.025),
        'negative_share_ci_high': np.quantile(bootstrap_negative, 0.975),
    }

bootstrap_rows = []
for region, subset in [('all', valid), *list(valid.groupby('_region'))]:
    row = cluster_bootstrap(subset)
    row['query_region'] = region
    bootstrap_rows.append(row)
bootstrap_intervals = pd.DataFrame(bootstrap_rows)
display(bootstrap_intervals.round(4))

## 6. Kalibrasi populasi opsional

Tahap ini tidak dijalankan otomatis. Isi template dengan populasi target resmi untuk Jakarta dan Banten. Gunakan sumber dengan definisi serta tahun yang sama, lalu ubah `verified` menjadi `True`.

Kalibrasi tetap merupakan **analisis sensitivitas strata kueri**, sebab `_region` bukan domisili penulis.

In [ ]:
reference_template = pd.DataFrame({
    'region': ['Jakarta', 'Banten'],
    'target_population': [np.nan, np.nan],
    'reference_year': [np.nan, np.nan],
    'source_name': ['', ''],
    'source_url': ['', ''],
    'verified': [False, False],
})
template_path = OUTPUT_DIR / 'regional_reference_template.csv'
reference_template.to_csv(template_path, index=False)

population_calibration_status = 'skipped_no_verified_reference'
population_sensitivity = pd.DataFrame()

if REFERENCE_PATH.exists():
    reference = pd.read_csv(REFERENCE_PATH)
    needed = {'region', 'target_population', 'reference_year', 'source_name', 'source_url', 'verified'}
    missing_reference = sorted(needed - set(reference.columns))
    if missing_reference:
        warnings.warn(f'Kalibrasi dilewati, kolom referensi kurang: {missing_reference}')
    else:
        reference['verified'] = reference['verified'].astype(str).str.lower().isin(['true', '1', 'yes'])
        reference['target_population'] = pd.to_numeric(reference['target_population'], errors='coerce')
        valid_reference = (
            reference['verified'].all()
            and reference['target_population'].gt(0).all()
            and reference['source_url'].fillna('').str.startswith(('http://', 'https://')).all()
            and set(valid['_region'].unique()).issubset(set(reference['region']))
        )
        if valid_reference:
            target = reference.set_index('region')['target_population']
            target_share = target / target.sum()
            observed = valid.groupby('_region')['weight_operational'].sum()
            observed_share = observed / observed.sum()
            factors = (target_share / observed_share).to_dict()
            valid['population_sensitivity_raw'] = valid['weight_operational'] * valid['_region'].map(factors)
            valid['weight_population_sensitivity'], pop_lo, pop_hi = trim_and_normalize(valid['population_sensitivity_raw'])
            population_rows = []
            for region, subset in valid.groupby('_region'):
                population_rows.append({
                    '_region': region,
                    'n_tweets': len(subset),
                    'target_population': target[region],
                    'calibration_factor': factors[region],
                    'mean_sentiment': weighted_mean(
                        subset['sentiment_value'], subset['weight_population_sensitivity']
                    ),
                })
            population_sensitivity = pd.DataFrame(population_rows)
            population_calibration_status = 'applied_verified_reference'
        else:
            warnings.warn('Kalibrasi dilewati: referensi belum lengkap, tidak terverifikasi, atau URL sumber kosong.')

print(f'Status kalibrasi populasi: {population_calibration_status}')
print(f'Template referensi: {template_path}')
if not population_sensitivity.empty:
    display(population_sensitivity.round(4))

## 7. Gabungkan hasil topik

Probabilitas topik dirata-ratakan dengan bobot operasional. Tweet yang tidak masuk korpus LDA tidak dipaksakan memperoleh topik.

In [ ]:
weighted_topic_estimates = pd.DataFrame()
if lda_path is not None:
    lda = pd.read_csv(lda_path, low_memory=False)
    lda['id'] = lda['id'].astype(str)
    probability_columns = [c for c in lda.columns if c.startswith('topic_') and c.endswith('_probability')]
    topic_data = valid[['id', '_region', 'weight_operational']].merge(
        lda[['id', *probability_columns]].drop_duplicates('id'), on='id', how='inner'
    )
    topic_rows = []
    for region, subset in [('all', topic_data), *list(topic_data.groupby('_region'))]:
        for column in probability_columns:
            topic_rows.append({
                'query_region': region,
                'topic_probability_column': column,
                'n_tweets': len(subset),
                'weighted_mean_probability': weighted_mean(subset[column], subset['weight_operational']),
            })
    weighted_topic_estimates = pd.DataFrame(topic_rows)
    display(weighted_topic_estimates.round(4))
else:
    print('Hasil LDA tidak ditemukan; ringkasan topik dilewati.')

## 8. Visualisasi dan ekspor

Seluruh keluaran tahap 05 disimpan dalam satu direktori. Metadata mencatat parameter agar analisis dapat direplikasi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(valid['weight_operational'], bins=35, color='#176B87', edgecolor='white')
axes[0].set(title='Distribusi bobot operasional', xlabel='Bobot ternormalisasi', ylabel='Jumlah tweet')

region_shares = valid.groupby('_region').agg(
    unweighted=('id', 'size'),
    weighted=('weight_operational', 'sum'),
)
region_shares = region_shares.div(region_shares.sum(axis=0), axis=1)
region_shares.plot(kind='bar', ax=axes[1], color=['#9CA3AF', '#D97706'])
axes[1].set(title='Kontribusi strata kueri', xlabel='Wilayah kueri', ylabel='Proporsi')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Tanpa bobot', 'Bobot operasional'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'weight_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

valid.to_csv(OUTPUT_DIR / 'weighted_analysis_results.csv', index=False)
weight_diagnostics.to_csv(OUTPUT_DIR / 'weight_diagnostics.csv', index=False)
coverage_by_region.to_csv(OUTPUT_DIR / 'coverage_by_region.csv', index=False)
coverage_by_keyword.to_csv(OUTPUT_DIR / 'coverage_by_keyword.csv', index=False)
account_contribution.to_csv(OUTPUT_DIR / 'account_contribution.csv', index=False)
sentiment_estimates.to_csv(OUTPUT_DIR / 'weighted_sentiment_estimates.csv', index=False)
bootstrap_intervals.to_csv(OUTPUT_DIR / 'cluster_bootstrap_intervals.csv', index=False)
if not weighted_topic_estimates.empty:
    weighted_topic_estimates.to_csv(OUTPUT_DIR / 'weighted_topic_estimates.csv', index=False)
if not population_sensitivity.empty:
    population_sensitivity.to_csv(OUTPUT_DIR / 'population_sensitivity.csv', index=False)

metadata = {
    'input_sentiment': str(sentiment_path),
    'input_lda': str(lda_path) if lda_path else None,
    'n_input_rows': int(len(df)),
    'n_analysis_rows': int(len(valid)),
    'n_unique_accounts': int(valid['analysis_account_id'].nunique()),
    'bootstrap_iterations': BOOTSTRAP_ITERATIONS,
    'random_state': RANDOM_STATE,
    'trim_quantiles': [LOWER_TRIM_QUANTILE, UPPER_TRIM_QUANTILE],
    'population_calibration_status': population_calibration_status,
    'methodological_scope': (
        'Mengurangi ketimpangan teramati dalam sampel X; tidak menginferensi demografi '
        'individu dan tidak menjamin representativitas populasi.'
    ),
}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Output tersimpan di: {OUTPUT_DIR}')
print('PENTING: hasil berbobot tetap menggambarkan percakapan X yang terkumpul, bukan opini seluruh penduduk.')